In [1]:
%pip install -q "transformers==4.57.1" "accelerate==1.11.0" "peft==0.17.1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
import gc
import hashlib
import re
import unicodedata
from pathlib import Path
from decimal import Decimal
import json
import torch

import pandas as pd
from peft import PeftModel
from PIL import Image
from collections import Counter
from safetensors.torch import load_file
from transformers import GenerationConfig, Qwen3VLForConditionalGeneration, AutoProcessor

In [4]:
MLLM = Path("/content/drive/MyDrive/MLLM/mordern_llm")
D_ROOT = Path("/content/drive/MyDrive/MLLM/section_d")
D_DATA = D_ROOT / "data"
D_DATA.mkdir(parents=True, exist_ok=True)

C_DATA = MLLM / "data/sroie"
manifest = json.loads((C_DATA / "manifest.json").read_text(encoding="utf-8"))
task = json.loads((C_DATA / "task_config.json").read_text(encoding="utf-8"))

TRAIN = manifest["train"]
FIELDS = tuple(task["fields"])
DOC_PROMPT = task["prompt"]
IMAGE_KWARGS = task["image_kwargs"]

processor = AutoProcessor.from_pretrained(MLLM / "outputs/qlora_run1/best_adapter", use_fast=False)
processor.tokenizer.padding_side = "right"

print("Section C folder:", MLLM)
print("Train / validation:", len(TRAIN), len(manifest["val"]))
print("Fields:", FIELDS)

Section C folder: /content/drive/MyDrive/MLLM/mordern_llm
Train / validation: 551 61
Fields: ('company', 'date', 'total')


In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

PILOT = D_ROOT / "outputs/sft_pilot_v1"
OUT = D_ROOT / "outputs/day4_heldout"
OUT.mkdir(parents=True, exist_ok=True)

recipe_path = PILOT / "recipe.json"
lock_path = D_ROOT / "data/heldout_manifest.json"

recipe = read_json(recipe_path)
heldout = read_json(lock_path)
TEST = heldout["test"]
TEST_IDS = [r["id"] for r in TEST]

used = manifest["train"] + manifest["val"] + manifest["test"]
used_ids = {r["id"] for r in used}
used_hashes = {r["sha256"] for r in used}
test_hashes = {r["sha256"] for r in TEST}

task = recipe["task"]
FIELDS = tuple(task["fields"])
DOC_PROMPT = task["prompt"]
EVAL_BUDGET = recipe["image_kwargs"]

DTYPE = {
    "torch.float16": torch.float16,
    "torch.bfloat16": torch.bfloat16
}[recipe["dtype"]]

DEVICE = torch.device("cuda:0")
ADAPTER = PILOT / "llm_lora/best_adapter"
MERGER = PILOT / "projector_only/best_merger.safetensors"
CHECKPOINT_STEP = {
    name: read_json(PILOT / name / "best_validation.json")["step"]
    for name in ("llm_lora", "projector_only")
}

print("Held-out receipts:", len(TEST))
print("Selected steps:", CHECKPOINT_STEP)
print("Dtype:", DTYPE)

Held-out receipts: 30
Selected steps: {'llm_lora': 8, 'projector_only': 8}
Dtype: torch.float16


In [6]:
def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(unicodedata.normalize("NFKC", value).casefold().split())
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?", text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")


def score_doc(answer, reference):
    raw_valid = True
    used_cleanup = False

    try:
        prediction = json.loads(answer)
    except json.JSONDecodeError:
        raw_valid = False
        prediction = None

        fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", answer.strip(), flags=re.IGNORECASE | re.DOTALL)

        if fence:
            used_cleanup = True
            try:
                prediction = json.loads(fence.group(1))
            except json.JSONDecodeError:
                pass

    schema_valid = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(value is None or isinstance(value, str) for value in prediction.values())
    )

    flags = {}

    for field in FIELDS:
        correct = False

        if schema_valid:
            predicted = prediction[field]
            expected = reference[field]

            if predicted is None or expected is None:
                correct = predicted is None and expected is None
            else:
                normalized = normal(field, predicted)
                correct = (
                    normalized is not None
                    and normalized == normal(field, expected)
                )

        flags[f"{field}_correct"] = correct

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema_valid,
        "used_fence_cleanup": used_cleanup,
        **flags,
        "all_correct": all(flags.values())
    }

In [12]:
def load_eval_model(setting):
    net = Qwen3VLForConditionalGeneration.from_pretrained(
        recipe["model_id"],
        revision=recipe["revision"],
        dtype=DTYPE,
        device_map={"": 0},
        attn_implementation="sdpa"
    )
    
    if setting == "llm_lora":
        net = PeftModel.from_pretrained(net, ADAPTER, is_trainable=False)
    elif setting == "projector_only":
        merger = net.get_submodule(recipe["projector"]).float()
        merger.load_state_dict(load_file(str(MERGER)), strict=True)
    elif setting != "base":
        raise ValueError(setting)

    return net.eval()


generation = GenerationConfig(
    do_sample=False,
    use_cache=True,
    max_new_tokens=task["max_new_tokens"],
    eos_token_id=processor.tokenizer.eos_token_id,
    pad_token_id=processor.tokenizer.pad_token_id
)

@torch.no_grad()
def predict_receipt(net, record):
    messages = [{
        "role" : "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": DOC_PROMPT}
        ]
    }]
    
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with Image.open(D_ROOT / record["image_path"]) as source:
        image = source.convert("RGB")
    
    batch = processor(
        text=[text],
        images=[image],
        images_kwargs=EVAL_BUDGET,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(DEVICE)
    
    with torch.autocast("cuda", dtype=DTYPE):
        output = net.generate(
            **batch, generation_config=generation, use_model_defaults=False
        )
    
    new_ids = output[0, batch["input_ids"].shape[1]:]
    return processor.tokenizer.decode(new_ids, skip_special_tokens=True).strip()

**Evaluate 30 receipt**

In [8]:
def sha256_file(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

weight_files = {
    "base": [],
    "llm_lora" : [
        ADAPTER / "adapter_config.json",
        ADAPTER / "adapter_model.safetensors"
    ],
    "projector_only": [MERGER]
}

def evaluate_setting(setting):
    path = OUT / f"{setting}.json"
    signature = {
        "recipe": sha256_file(recipe_path),
        "test_lock": sha256_file(lock_path),
        "weights": [sha256_file(p) for p in weight_files[setting]]
    }
    
    state = (read_json(path) if path.exists() else {"signature": signature, "rows": []})
    
    rows = state["rows"]
    if len(rows) == len(TEST):
        print(f"{setting}: loaded all 30 saved results")
        return rows
    
    net = load_eval_model(setting)
    try:
        for record in TEST[len(rows):]:
            answer = predict_receipt(net, record)
            rows.append({
                "id": record["id"],
                "reference": record["reference"],
                "answer": answer,
                **score_doc(answer, record["reference"])
            })

            if len(rows) % 5 == 0 or len(rows) == len(TEST):
                tmp = path.with_suffix(".tmp")
                tmp.write_text(json.dumps(state, ensure_ascii=False), encoding="utf-8")
                tmp.replace(path)
                print(f"{setting}: {len(rows)}/{len(TEST)} saved", flush=True)
    finally:
        del net
        gc.collect()
        torch.cuda.empty_cache()

    return rows

In [13]:
results = {}
for setting in ("base", "llm_lora", "projector_only"):
    results[setting] = evaluate_setting(setting)

base: 5/30 saved
base: 10/30 saved
base: 15/30 saved
base: 20/30 saved
base: 25/30 saved
base: 30/30 saved
llm_lora: 5/30 saved
llm_lora: 10/30 saved
llm_lora: 15/30 saved
llm_lora: 20/30 saved
llm_lora: 25/30 saved
llm_lora: 30/30 saved
projector_only: 5/30 saved
projector_only: 10/30 saved
projector_only: 15/30 saved
projector_only: 20/30 saved
projector_only: 25/30 saved
projector_only: 30/30 saved


**Metrics table**

In [14]:
metrics = [
    "company_correct", "date_correct", "total_correct",
    "all_correct", "raw_json_valid", "schema_valid"
]

summary = []
for setting, rows in results.items():
    assert len(rows) == 30
    summary.append({
        "setting": setting,
        "step": CHECKPOINT_STEP.get(setting, "-"),
        "all_correct_count": sum(r["all_correct"] for r in rows),
        **{
            f"{key}_pct": 100 * sum(r[key] for r in rows) / len(rows)
            for key in metrics
        }
    })

summary = pd.DataFrame(summary)
display(summary.round(2))
summary.to_csv(OUT / "summary.csv", index=False)

,setting,step,all_correct_count,company_correct_pct,date_correct_pct,total_correct_pct,all_correct_pct,raw_json_valid_pct,schema_valid_pct
0,base,-,19,86.67,83.33,86.67,63.33,100.0,100.0
1,llm_lora,8,21,90.00,90.00,86.67,70.00,100.0,100.0
2,projector_only,8,15,70.00,90.00,83.33,50.00,100.0,100.0


In [15]:
base_by_id = {r["id"]: r for r in results["base"]}
comparison = []

for setting in ("llm_lora", "projector_only"):
    trained_by_id = {r["id"]: r for r in results[setting]}
    assert base_by_id.keys() == trained_by_id.keys()

    counts = Counter()
    examples = {}

    for sample_id, before in base_by_id.items():
        after = trained_by_id[sample_id]

        if before["all_correct"]:
            outcome = "both_correct" if after["all_correct"] else "lost"
        else:
            outcome = "gained" if after["all_correct"] else "both_wrong"

        counts[outcome] += 1
        examples.setdefault(outcome, (sample_id, before, after))

        if (outcome == "both_correct" and not before["raw_json_valid"] and after["raw_json_valid"]):
            counts["raw_format_fixed_both_correct"] += 1

    comparison.append({
        "setting": setting,
        **{key: counts[key] for key in (
            "both_correct", "gained", "lost", "both_wrong",
            "raw_format_fixed_both_correct"
        )},
        "change_pp": 100 * (counts["gained"] - counts["lost"]) / len(TEST)
    })

    for outcome in ("gained", "lost"):
        if outcome in examples:
            sample_id, before, after = examples[outcome]
            print(f"\n{setting} | {outcome} | {sample_id}")
            print("Reference:", before["reference"])
            print("Base answer:", before["answer"])
            print("Trained answer:", after["answer"])
            

comparison = pd.DataFrame(comparison)
display(comparison.round(2))
comparison.to_csv(OUT / "paired_summary.csv", index=False)


llm_lora | gained | X51005442375
Reference: {'company': 'UNIHAKKA INTERNATIONAL SDN BHD', 'date': '25 MAR 2018', 'total': '$8.50'}
Base answer: {
  "company": "UNIHAKKA INTERNATIONAL SDN BHD",
  "date": "25 Mar 2018 18:26",
  "total": "8.50"
}
Trained answer: {"company": "UNIHAKKA INTERNATIONAL SDN BHD", "date": "25 Mar 2018", "total": "8.50"}

projector_only | gained | X51005442375
Reference: {'company': 'UNIHAKKA INTERNATIONAL SDN BHD', 'date': '25 MAR 2018', 'total': '$8.50'}
Base answer: {
  "company": "UNIHAKKA INTERNATIONAL SDN BHD",
  "date": "25 Mar 2018 18:26",
  "total": "8.50"
}
Trained answer: {"company": "UNIHAKKA INTERNATIONAL SDN BHD", "date": "25 MAR 2018", "total": "$8.50"}

projector_only | lost | X51005719855
Reference: {'company': 'CPI ROCKU SDN. BHD.', 'date': '19/02/2018', 'total': '181.55'}
Base answer: {
  "company": "CPI ROCKU SDN. BHD.",
  "date": "19/02/2018",
  "total": "181.55"
}
Trained answer: {"company": "ROCKU YAKINIKU C.P.I ROCKU SDN. BHD. (1096317-P)

,setting,both_correct,gained,lost,both_wrong,raw_format_fixed_both_correct,change_pp
0,llm_lora,19,2,0,9,0,6.67
1,projector_only,13,2,6,9,0,-13.33
